# Project 23: News Summariser with Evaluation

**Question:** Does a pretrained model (BART) write better news summaries than a simple Lead-3 baseline, measured by ROUGE and human ratings?

## 0. Install (run once)

In [1]:
!pip install datasets transformers torch rouge-score nltk pandas

## 1. Imports and settings

In [4]:
import random
import pandas as pd
import nltk
from datasets import load_dataset
from transformers import pipeline
from rouge_score import rouge_scorer

nltk.download("punkt")
nltk.download("punkt_tab")

SEED = 42
random.seed(SEED)

N = 50   # number of articles. 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\elppa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\elppa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 2. Load data (CNN/DailyMail: news article + human-written summary)

In [5]:
data = load_dataset("abisee/cnn_dailymail", "3.0.0", split="test").shuffle(seed=SEED).select(range(N))

df = pd.DataFrame({
    "article": data["article"],
    "reference": data["highlights"],   # the human summary
})
df.head(3)

,article,reference
0,(CNN) I see signs of a revolution everywhere. ...,CNN's Dr. Sanjay Gupta says we should legalize...
1,He looks barely teenage. But this child has am...,Child has amassed thousands of Twitter followe...
2,New Jersey Governor Chris Christie wasn't look...,The presidential hopeful held a town hall meet...


## 3. Baseline: Lead-3 (first 3 sentences of the article)

In [6]:
def lead3(text):
    sentences = nltk.sent_tokenize(text)
    return " ".join(sentences[:3])

df["lead3"] = df["article"].apply(lead3)
df[["reference", "lead3"]].head(2)

,reference,lead3
0,CNN's Dr. Sanjay Gupta says we should legalize...,(CNN) I see signs of a revolution everywhere. ...
1,Child has amassed thousands of Twitter followe...,He looks barely teenage. But this child has am...


## 4. Main model: BART (pretrained for news summarising)

In [7]:
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# If too slow, use "sshleifer/distilbart-cnn-12-6" instead
model_name = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.eval()

def bart_summary(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=130,
            min_length=30,
            num_beams=4,
            do_sample=False,
            early_stopping=True,
        )
    return tokenizer.decode(ids[0], skip_special_tokens=True)

df["bart"] = [bart_summary(a) for a in tqdm(df["article"])]
df[["reference", "bart"]].head(2)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

,reference,bart
0,CNN's Dr. Sanjay Gupta says we should legalize...,CNN's John Sutter says he sees signs of a medi...
1,Child has amassed thousands of Twitter followe...,"Baby-faced boy from Memphis, Tennessee, has am..."


## 5. ROUGE scores

In [8]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def add_rouge(df, col):
    scores = [scorer.score(ref, pred) for ref, pred in zip(df["reference"], df[col])]
    df[f"{col}_r1"] = [s["rouge1"].fmeasure for s in scores]
    df[f"{col}_r2"] = [s["rouge2"].fmeasure for s in scores]
    df[f"{col}_rL"] = [s["rougeL"].fmeasure for s in scores]

add_rouge(df, "lead3")
add_rouge(df, "bart")

results = pd.DataFrame({
    "method": ["Lead-3 (baseline)", "BART"],
    "ROUGE-1": [df["lead3_r1"].mean(), df["bart_r1"].mean()],
    "ROUGE-2": [df["lead3_r2"].mean(), df["bart_r2"].mean()],
    "ROUGE-L": [df["lead3_rL"].mean(), df["bart_rL"].mean()],
}).round(4)
results

,method,ROUGE-1,ROUGE-2,ROUGE-L
0,Lead-3 (baseline),0.3845,0.1502,0.2370
1,BART,0.4191,0.1756,0.2796


## 6. Human rating file

Rate each summary from 1 (bad) to 5 (very good).
- **relevance**: does it keep the main points?
- **fluency**: does it read well?
- **accuracy**: are the facts correct (no made-up facts)?

In [9]:
sample = df.sample(20, random_state=SEED).reset_index()

rows = []
for _, r in sample.iterrows():
    for method in ["lead3", "bart"]:
        rows.append({
            "article_id": r["index"],
            "method": method,
            "reference": r["reference"],
            "summary": r[method],
        })

sheet = pd.DataFrame(rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet.insert(0, "row_id", range(1, len(sheet) + 1))

# 1) Answer key: keep this PRIVATE, do not give to raters
sheet[["row_id", "article_id", "method"]].to_csv("answer_key.csv", index=False)

# 2) Sheet for raters: no method column
for_raters = sheet[["row_id", "reference", "summary"]].copy()
for_raters["relevance"] = ""
for_raters["fluency"] = ""
for_raters["accuracy"] = ""

from openpyxl.styles import Alignment
with pd.ExcelWriter("human_rating_sheet_for_raters.xlsx", engine="openpyxl") as w:
    for_raters.to_excel(w, index=False, sheet_name="Sheet1")
    ws = w.sheets["Sheet1"]
    ws.column_dimensions["B"].width = 80
    ws.column_dimensions["C"].width = 80
    for col in ["B", "C"]:
        for cell in ws[col]:
            cell.alignment = Alignment(wrap_text=True, vertical="top")

print("Saved: answer_key.csv (private) and human_rating_sheet_for_raters.xlsx")

Saved: answer_key.csv (private) and human_rating_sheet_for_raters.xlsx


## 7. Error analysis (20 marks)

Look at the 10 worst BART summaries. For each one, write WHY it failed. Common reasons:
missing key info, made-up facts (hallucination), repeated sentences, wrong focus, too short or too long.

In [10]:
worst = df.sort_values("bart_rL").head(10)

for i, row in worst.iterrows():
    print("=" * 80)
    print("ROUGE-L:", round(row["bart_rL"], 3))
    print("\nREFERENCE:", row["reference"])
    print("\nBART:", row["bart"])
    print("\nARTICLE START:", row["article"][:400])

ROUGE-L: 0.144

REFERENCE: YouGov poll of Scots: 49% backing SNP, 25% Labour,  18% Conservatives .
SNP leader Nicola Sturgeon offered to prop up Miliband in government .
But she is demanding full fiscal autonomy as a price for her support .
Miliband warns it would create a £7billion blackhole in Scottish finances .

BART: Labour leader uses speech in Edinburgh to claim Tory election campaign has descended into 'desperation and panic', as some polls put Labour ahead across the UK. Mr Miliband rocked by survey showing that in Scotland, 49 per cent of people plan to vote for the SNP, with just 25 per cent backing Labour. In September's referendum, 55 per cent voted in favour of staying in the UK, but the SNP has seen a surge in support since then.

ARTICLE START: Ed Miliband today rushed to Scotland to try to shore up his hope of becoming Prime Minister as a damning poll showed almost half of Scots are ready to back the SNP. The Labour leader used a speech in Edinburgh to claim the Tory e

In [12]:
error_table = pd.DataFrame({
    "topic": [
        "Miliband / SNP poll", "Corinthian Colleges", "Hillsborough photos",
        "Cassey Ho video", "Obama family hike", "Barcelona vs PSG",
        "NYC subway fire", "Peru mummy", "Mayweather's chef", "Memphis boy on Twitter"
    ],
    "rouge_l": [0.144, 0.152, 0.159, 0.171, 0.186, 0.190, 0.190, 0.198, 0.198, 0.202],
    "error_type": [
        "different focus; key point missing",
        "possible wrong name; key point missing",
        "different focus",
        "different focus",
        "different focus; copies article sentences; minor detail",
        "key point missing; minor detail",
        "different focus; copies article sentences",
        "different focus",
        "key point missing; copies article sentences",
        "different focus; copies article sentences; unsuitable detail",
    ],
    "notes": [
        "Leaves out Sturgeon's offer of support and the fiscal autonomy demand",
        "Names Gloria Borger, reference says David Wheeler (verify in article)",
        "Accurate, but leaves out phone number, website and Home Office inquiry",
        "Good summary, skips 2 million subscribers and weight comments",
        "Misses reporters being sent home; adds golf detail",
        "Never says Barcelona reached the semi-final; adds offside detail",
        "Explains the cause, not the outcome (no injuries)",
        "Leaves out Chan Chan and that cleaners called police",
        "Never says what the weaknesses are (Twizzlers, noodles, hotdogs)",
        "Quotes offensive tweets; misses the gun video",
    ],
})
error_table

,topic,rouge_l,error_type,notes
0,Miliband / SNP poll,0.144,different focus; key point missing,Leaves out Sturgeon's offer of support and the...
1,Corinthian Colleges,0.152,possible wrong name; key point missing,"Names Gloria Borger, reference says David Whee..."
2,Hillsborough photos,0.159,different focus,"Accurate, but leaves out phone number, website..."
3,Cassey Ho video,0.171,different focus,"Good summary, skips 2 million subscribers and ..."
4,Obama family hike,0.186,different focus; copies article sentences; min...,Misses reporters being sent home; adds golf de...
5,Barcelona vs PSG,0.190,key point missing; minor detail,Never says Barcelona reached the semi-final; a...
6,NYC subway fire,0.190,different focus; copies article sentences,"Explains the cause, not the outcome (no injuries)"
7,Peru mummy,0.198,different focus,Leaves out Chan Chan and that cleaners called ...
8,Mayweather's chef,0.198,key point missing; copies article sentences,"Never says what the weaknesses are (Twizzlers,..."
9,Memphis boy on Twitter,0.202,different focus; copies article sentences; uns...,Quotes offensive tweets; misses the gun video


In [13]:
error_table["error_type"].str.split("; ").explode().value_counts()

error_type
different focus             7
key point missing           4
copies article sentences    4
minor detail                2
possible wrong name         1
unsuitable detail           1
Name: count, dtype: int64

In [14]:
from importlib.metadata import version

packages = ["transformers", "torch", "datasets", "rouge-score", "nltk",
            "pandas", "scipy", "openpyxl", "tqdm"]

with open("requirements.txt", "w") as f:
    for p in packages:
        f.write(f"{p}=={version(p)}\n")

print(open("requirements.txt").read())

transformers==5.23.0
torch==2.15.0
datasets==5.2.0
rouge-score==0.1.2
nltk==3.10.0
pandas==3.0.3
scipy==1.18.0
openpyxl==3.1.5
tqdm==4.68.2



In [15]:
df[["article", "reference", "lead3", "bart"]].to_csv("sample_50_articles.csv", index=False, encoding="utf-8-sig")
print("Saved sample_50_articles.csv")

Saved sample_50_articles.csv
